In [1]:
!pip install llama-index==0.10

Defaulting to user installation because normal site-packages is not writeable
   ---------------------------------------- 0.0/1.6 MB ? eta -:--:--
   ---------------------------------------- 1.6/1.6 MB 7.6 MB/s  0:00:00
   ---------------------------------------- 0.0/1.2 MB ? eta -:--:--
   ---------------------------------------- 1.2/1.2 MB 8.5 MB/s  0:00:00
   ---------------------------------------- 0.0/15.8 MB ? eta -:--:--
   ----- ---------------------------------- 2.1/15.8 MB 9.8 MB/s eta 0:00:02
   ----------- ---------------------------- 4.7/15.8 MB 11.0 MB/s eta 0:00:02
   --------------- ------------------------ 6.3/15.8 MB 9.9 MB/s eta 0:00:01
   --------------------- ------------------ 8.4/15.8 MB 10.0 MB/s eta 0:00:01
   --------------------------- ------------ 11.0/15.8 MB 10.3 MB/s eta 0:00:01
   --------------------------------- ------ 13.4/15.8 MB 10.5 MB/s eta 0:00:01
   ---------------------------------------  15.5/15.8 MB 10.5 MB/s eta 0:00:01
   ------------------

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
thinc 8.3.6 requires numpy<3.0.0,>=2.0.0, but you have numpy 1.26.4 which is incompatible.


In [2]:
!pip install python-dotenv

Defaulting to user installation because normal site-packages is not writeable


In [4]:
import os
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv())

True

In [6]:
api_key = os.environ.get('OPENAI_API_KEY')

In [28]:
#Step 2: Define the embedding mode, LLM, and data
from llama_index.embeddings.openai import OpenAIEmbedding
from llama_index.llms.openai import OpenAI
from llama_index.core.settings import Settings

Settings.llm = OpenAI(model='gpt-4', temperature=0.1)
Settings.embed_model = OpenAIEmbedding()

In [29]:
from llama_index.core import SimpleDirectoryReader

#Load Data
documents = SimpleDirectoryReader(
        input_files=[r"C:\Users\sampa\OneDrive\Documents\2025\Job Hunting\Thomas Reuters AI ML Job posting.txt"]
).load_data()

In [17]:
#Step3: Index Data and set up the query engine
from llama_index.core.node_parser import SimpleNodeParser

node_parser = SimpleNodeParser.from_defaults(chunk_size =1024)

#Extract Nodes from documents
nodes = node_parser.get_nodes_from_documents(documents)

In [19]:
#Display first two lines for first two nodes
for i in range(2):
    node_content = nodes[i].text
    first_two_lines = "\n".join(node_content.splitlines()[:2])
    print(f"Node {i + 1}: \n{first_two_lines}\n")

Node 1: 
Are you passionate about the chance to bring your data quality improvement experience to a world class organization that is leading the way in both content and technology to serve and protect our citizens home and abroad? Do you have the skills necessary to manage, understand, and analyze inhouse and customer data including text mining, developing predictive systems, risk scoring, creating efficient algorithms, data quality improvement and other related activities? Then Thomson Reuters Special Services (TRSS) is looking for you!


Node 2: 
for building and deploying applications
Experience in a fast-paced, agile environment managing uncertainty and ambiguity.



In [20]:
!pip install weaviate-client llama-index-vector-stores-weaviate

Defaulting to user installation because normal site-packages is not writeable
   ---------------------------------------- 0.0/579.1 kB ? eta -:--:--
   ---------------------------------------- 579.1/579.1 kB 6.7 MB/s  0:00:00
   ---------------------------------------- 0.0/4.5 MB ? eta -:--:--
   ----------- ---------------------------- 1.3/4.5 MB 6.1 MB/s eta 0:00:01
   -------------------- ------------------- 2.4/4.5 MB 5.4 MB/s eta 0:00:01
   ------------------------------------- -- 4.2/4.5 MB 7.0 MB/s eta 0:00:01
   ---------------------------------------- 4.5/4.5 MB 6.6 MB/s  0:00:00
   ---------------------------------------- 0.0/7.6 MB ? eta -:--:--
   ----------- ---------------------------- 2.1/7.6 MB 10.7 MB/s eta 0:00:01
   ----------------- ---------------------- 3.4/7.6 MB 9.2 MB/s eta 0:00:01
   ----------------------- ---------------- 4.5/7.6 MB 7.5 MB/s eta 0:00:01
   --------------------------------- ------ 6.3/7.6 MB 7.6 MB/s eta 0:00:01
   ---------------------------

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
llama-index 0.10.0 requires llama-index-core<0.11.0,>=0.10.0, but you have llama-index-core 0.14.0 which is incompatible.
llama-index-agent-openai 0.1.7 requires llama-index-core<0.11.0,>=0.10.1, but you have llama-index-core 0.14.0 which is incompatible.
llama-index-embeddings-openai 0.1.11 requires llama-index-core<0.11.0,>=0.10.1, but you have llama-index-core 0.14.0 which is incompatible.
llama-index-llms-openai 0.1.31 requires llama-index-core<0.11.0,>=0.10.57, but you have llama-index-core 0.14.0 which is incompatible.
llama-index-multi-modal-llms-openai 0.1.9 requires llama-index-core<0.11.0,>=0.10.1, but you have llama-index-core 0.14.0 which is incompatible.
llama-index-program-openai 0.1.7 requires llama-index-core<0.11.0,>=0.10.57, but you have llama-index-core 0.14.0 which is incompatible.
llama-index-

In [24]:
import weaviate
import os

headers = {
    "X-OpenAI-Api-Key": os.getenv("OPENAI_API_KEY")
}

client = weaviate.connect_to_local(headers=headers)

In [25]:
from llama_index.core import VectorStoreIndex, StorageContext
from llama_index.vector_stores.weaviate import WeaviateVectorStore

index_name = "JobHuntingTR"

vector_store = WeaviateVectorStore(
    weaviate_client = client,
    index_name = index_name
)

#Set up storage for the embeddings
storage_context = StorageContext.from_defaults(vector_store=vector_store)


In [26]:
#Setup the index
#Build VectoreStoreIndex that takes care of chunking documents
# and encoding chunks to embeddings for future retrieval

index = VectorStoreIndex(
    nodes,
    storage_context = storage_context,
)

In [50]:
#The QueryEngine class is equipped with the generator
# and facilitates the retrieval and generation steps
query_engine = index.as_query_engine()

In [51]:
#Run you naive RAG query
response = query_engine.query(
    "What are top 5 skills needed for Thomson Reuters position?"
)

In [52]:
response.response

'The top five skills needed for the Thomson Reuters position are:\n\n1. Experience in building and deploying applications.\n2. Proficiency in Python and experience delivering minimum viable products in a large enterprise environment.\n3. Experience developing libraries and APIs for others to use.\n4. Outstanding communication and data-driven decision-making collaboration with Product + Business Stakeholders.\n5. Ability to manage uncertainty and ambiguity in a fast-paced, agile environment.'

In [53]:
#Retrieval Optimization with Hybrid Search
query_engine = index.as_query_engine(
    vector_store_query_mode="hybrid",
    alpha=0.9
)

In [54]:
response = query_engine.query(
    "What are top 5 skills needed for Thomson Reuters position?"
)

In [55]:
response.response

'The top five skills needed for the Thomson Reuters position are:\n\n1. Experience in building and deploying applications.\n2. Proficiency in Python and experience delivering minimum viable products in a large enterprise environment.\n3. Experience developing libraries and APIs for others to use.\n4. Outstanding communication and data-driven decision-making collaboration with Product + Business Stakeholders.\n5. Ability to manage uncertainty and ambiguity in a fast-paced, agile environment.'

In [40]:
#Implementing Re-ranking in the RAG pipeline
!pip install torch sentence-transformers

Defaulting to user installation because normal site-packages is not writeable
   ---------------------------------------- 0.0/241.4 MB ? eta -:--:--
   ---------------------------------------- 2.1/241.4 MB 10.7 MB/s eta 0:00:23
    --------------------------------------- 4.5/241.4 MB 10.7 MB/s eta 0:00:23
   - -------------------------------------- 6.3/241.4 MB 10.2 MB/s eta 0:00:24
   - -------------------------------------- 8.7/241.4 MB 10.5 MB/s eta 0:00:23
   - -------------------------------------- 10.7/241.4 MB 10.3 MB/s eta 0:00:23
   -- ------------------------------------- 12.8/241.4 MB 10.5 MB/s eta 0:00:22
   -- ------------------------------------- 14.7/241.4 MB 10.1 MB/s eta 0:00:23
   -- ------------------------------------- 16.8/241.4 MB 10.1 MB/s eta 0:00:23
   --- ------------------------------------ 19.1/241.4 MB 10.1 MB/s eta 0:00:22
   --- ------------------------------------ 21.5/241.4 MB 10.3 MB/s eta 0:00:22
   --- ------------------------------------ 22.8/241.4 

C:\Users\sampa\AppData\Roaming\Python\Python311\site-packages\IPython\utils\_process_common.py:117: ResourceWarning: unclosed file <_io.BufferedWriter name=3>
  return out
C:\Users\sampa\AppData\Roaming\Python\Python311\site-packages\IPython\utils\_process_common.py:117: ResourceWarning: unclosed file <_io.BufferedReader name=4>
  return out
C:\Users\sampa\AppData\Roaming\Python\Python311\site-packages\IPython\utils\_process_common.py:117: ResourceWarning: unclosed file <_io.BufferedReader name=5>
  return out


In [42]:
from llama_index.core.postprocessor import SentenceTransformerRerank

#Define reranker model
rerank = SentenceTransformerRerank(
    top_n = 2,
    model = "BAAI/bge-reranker-base"
)


config.json:   0%|          | 0.00/799 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.11G [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/443 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/279 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

In [56]:
#Add reranker to the query engine
query_engine = index.as_query_engine(
    similarity_top_k = 6,
    node_postprocessors = [rerank],
)

In [57]:
response = query_engine.query(
    "What are top 5 skills needed for Thomson Reuters position?"
)

In [58]:
response.response

'The top five skills needed for the Generative AI & Machine Learning Engineer position at Thomson Reuters are:\n\n1. Proficiency in Python\n2. At least 3+ years of practical, relevant experience building AI/ML products and applications, including recent experience using Generative AI technologies\n3. Solid software engineering skills and experience\n4. Experience as a technical leader, including aiding in ideation with product stakeholders, dealing with uncertainty and ambiguity in problem statements, breaking down complex problems, formulating research and development plans, and communicating progress and plans with varied stakeholders\n5. Hands-on coding experience on AI/ML projects in the current role and experience in designing, developing, and implementing machine learning models and algorithms.'